<a href="https://colab.research.google.com/github/sparshbansal-newton/deep-learning-labs/blob/main/Notebooks/7_pytorch_training_pipeline/pytorch_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔥 Building a Training Pipeline in PyTorch (from scratch)

In this notebook we build a **binary classifier from the ground up** using PyTorch's core building blocks and train a **Logistic Regression** model (a single-layer neural network) to predict whether a patient has diabetes, using the **Pima Indians Diabetes** dataset.

### 🗺️ The stages of (almost) every training pipeline
1. **Load & explore** the data
2. **Split** into train / test sets
3. **Preprocess** (feature scaling, prepare the target)
4. **Convert** NumPy arrays → PyTorch tensors
5. **Define the model** (parameters, forward pass, loss)
6. **Train** (the loop) and **evaluate**

> 💡 **Hint for students:** Read each markdown cell *before* running the code cell below it. Try to predict what the output shape or value will be, then check yourself.

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1️⃣ Load & explore the data

We use the classic **Pima Indians Diabetes** dataset. Each row is a patient with 8 medical measurements, and the target `Outcome` tells us whether they developed diabetes (`1`) or not (`0`).

The raw CSV has **no header row**, so we pass our own column names via `names=columns`.

| Feature | Meaning |
|---|---|
| `Pregnancies` | Number of times pregnant |
| `Glucose` | Plasma glucose concentration |
| `BloodPressure` | Diastolic blood pressure (mm Hg) |
| `SkinThickness` | Triceps skin fold thickness (mm) |
| `Insulin` | 2-Hour serum insulin (mu U/ml) |
| `BMI` | Body mass index |
| `DiabetesPedigreeFunction` | Diabetes likelihood from family history |
| `Age` | Age (years) |
| `Outcome` | **Target** → 1 = diabetes, 0 = no diabetes |

In [2]:
# The CSV has no header, so we supply the column names ourselves
columns = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
df = pd.read_csv(url, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [9]:
# (rows, columns) -> we expect 768 patients and 9 columns (8 features + 1 target)
df.shape

(768, 9)

In [10]:
# How many patients fall into each class?
# 0 = no diabetes, 1 = diabetes. This tells us whether the dataset is balanced.
df['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [11]:
# Quick statistical summary. Notice the very different scales:
# Glucose is in the 100s while DiabetesPedigreeFunction is < 3.
# This is exactly why we will need feature scaling later!
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


## 2️⃣ Train / test split

We hold out **20%** of the data as a **test set** the model never trains on. This is how we honestly measure whether the model *generalizes* to unseen patients instead of just memorizing.

- `X` = the 8 feature columns (everything except `Outcome`)
- `y` = the target column (`Outcome`)

> 💡 **Hint:** `df.iloc[:, :-1]` grabs *all columns except the last* (our features), and `df.iloc[:, -1]` grabs *only the last column* (our target). We set `random_state=42` so the split is **reproducible** — you'll get the same results every run.

In [12]:
# Features = all columns except the last; Target = the last column (Outcome)
X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, :-1],   # X: the 8 features
    df.iloc[:, -1],    # y: the Outcome
    test_size=0.2,
    random_state=42
)

print("Train samples:", X_train.shape[0])
print("Test samples: ", X_test.shape[0])

Train samples: 614
Test samples:  154


## 3️⃣ Feature scaling

Our features live on **wildly different scales** (`Glucose` ≈ 100s, `DiabetesPedigreeFunction` < 3). Gradient descent struggles when features have very different ranges — the large-scale features dominate the updates.

`StandardScaler` rescales each column to have **mean 0** and **standard deviation 1**.

 **Critical rule:** call `fit_transform` on the **training data only**, then `transform` (not `fit`!) on the test data. The scaler must learn its statistics *only* from the training set — otherwise information from the test set "leaks" into training. This is called **data leakage**.

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # learn mean/std from TRAIN, then scale train
X_test = scaler.transform(X_test)        # apply the SAME mean/std to test

In [14]:
# Notice the values are now centered around 0 (and this is a NumPy array, not a DataFrame)
X_train

array([[-0.52639686, -1.15139792, -3.75268255, ..., -4.13525578,
        -0.49073479, -1.03594038],
       [ 1.58804586, -0.27664283,  0.68034485, ..., -0.48916881,
         2.41502991,  1.48710085],
       [-0.82846011,  0.56687102, -1.2658623 , ..., -0.42452187,
         0.54916055, -0.94893896],
       ...,
       [ 1.8901091 , -0.62029661,  0.89659009, ...,  1.76054443,
         1.981245  ,  0.44308379],
       [-1.13052335,  0.62935353, -3.75268255, ...,  1.34680407,
        -0.78487662, -0.33992901],
       [-1.13052335,  0.12949347,  1.43720319, ..., -1.22614383,
        -0.61552223, -1.03594038]], shape=(614, 8))

In [15]:
# Our target is already numeric (0 or 1), but it's still a pandas Series
y_train

60     0
618    1
346    0
294    0
231    1
      ..
71     0
106    0
270    1
435    1
102    0
Name: Outcome, Length: 614, dtype: int64

### Prepare the target

The `Outcome` column is **already** `0`/`1`, so unlike a text label (e.g. `"M"`/`"B"`) we do **not** need a `LabelEncoder` here. We just pull the raw values out of the pandas Series into a plain NumPy array so we can turn it into a tensor next.

> 💡 **Hint:** `.values` (or `.to_numpy()`) converts a pandas Series/DataFrame into a NumPy array. If your target were text categories, *this* is where you'd call `LabelEncoder().fit_transform(...)` instead.

In [16]:
# Pull the 0/1 labels out as NumPy arrays
y_train = y_train.values
y_test = y_test.values

In [17]:
# Now it's a NumPy array of 0s and 1s
y_train

array([0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0,
       1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1,
       0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0,
       1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0,
       1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0,

## 4️⃣ NumPy arrays → PyTorch tensors

PyTorch does its math on **tensors**, not NumPy arrays. `torch.from_numpy(...)` wraps an array in a tensor **without copying the data**.

> 💡 **Hint:** dtype matters! `StandardScaler` outputs `float64`, so our feature tensors are `float64`. That's why later, in the model, we create the weights as `dtype=torch.float64` too — PyTorch throws a dtype-mismatch error if you multiply a `float32` tensor by a `float64` one.

In [18]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [19]:
# Expect: [num_train_samples, 8 features]
X_train_tensor.shape

torch.Size([614, 8])

In [20]:
# Expect: a 1-D tensor of labels, one per training sample
y_train_tensor.shape

torch.Size([614])

## 5️⃣ Defining the model

Our model is **logistic regression** — the simplest possible neural network (a single neuron). It has three pieces:

**1. Parameters (what the model *learns*)**
- `weights`: one number per feature → shape `(8, 1)`, started as small random values.
- `bias`: a single number, started at 0.
- Both have `requires_grad=True` so PyTorch's **autograd** tracks every operation and can compute gradients for us.

**2. Forward pass (making a prediction)**

$$z = X \cdot W + b \qquad \hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

The `sigmoid` squashes any real number into `(0, 1)`, which we read as *"probability of diabetes."*

**3. Loss function (how wrong we are) — Binary Cross-Entropy**

$$L = -\frac{1}{N}\sum \Big[ y \log(\hat{y}) + (1 - y)\log(1 - \hat{y}) \Big]$$

> 💡 **Hint — why the `clamp`?** If `y_pred` is ever exactly `0` or `1`, `log(0) = -∞` and the loss blows up to `NaN`. Clamping to `[1e-7, 1 - 1e-7]` keeps it numerically safe. Real libraries (`nn.BCELoss`) do this for you.

In [21]:
class MySimpleNN:

    def __init__(self, X):
        # LEARNABLE PARAMETERS
        # One weight per input feature -> shape (num_features, 1)
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        # A single bias term, started at 0
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        # z = X . W + b  (the linear part)
        z = torch.matmul(X, self.weights) + self.bias
        # sigmoid squashes z into a probability in (0, 1)
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_function(self, y_pred, y):
        # y_pred is shape (N, 1) but y is (N,). Reshape y to (N, 1) so the
        # element-wise math lines up, and cast to float for the log operations.
        y = y.view(-1, 1).double()

        # Clamp predictions to avoid log(0), which would be -infinity
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

        # Binary Cross-Entropy loss
        loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
        return loss

### Hyperparameters

These are knobs **we** choose (the model does not learn them):

- **`learning_rate`** — how big a step we take when updating parameters. Too big → training diverges; too small → training crawls.
- **`epochs`** — how many full passes over the training data we make.

> 💡 **Hint:** Once everything runs, come back and experiment! Try `learning_rate = 0.5` or `epochs = 100` and watch how the loss changes.

In [22]:
learning_rate = 0.1
epochs = 50

## 6️⃣ The training loop

This is the beating heart of deep learning. Every epoch repeats the **same 5 steps**:

1. **Forward pass** → compute predictions `y_pred`.
2. **Compute loss** → how wrong were we?
3. **Backward pass** → `loss.backward()` uses autograd to fill in `.grad` for every parameter (this is **backpropagation**!).
4. **Update parameters** → nudge weights & bias *against* the gradient (gradient **descent**). Wrapped in `torch.no_grad()` so this bookkeeping isn't itself tracked.
5. **Zero the gradients** → PyTorch *accumulates* gradients by default, so we reset them to 0 before the next epoch.

> ⚠️ **Common bug:** forgetting `.grad.zero_()`. If you skip it, gradients from every previous epoch add together and training goes haywire.

> 💡 **Watch:** the printed loss should **steadily decrease** — that's your proof the model is learning.

In [23]:
# create model (this initializes the random weights + zero bias)
model = MySimpleNN(X_train_tensor)

# training loop
for epoch in range(epochs):

    # 1) forward pass -> predictions
    y_pred = model.forward(X_train_tensor)

    # 2) compute how wrong we are
    loss = model.loss_function(y_pred, y_train_tensor)

    # 3) backward pass -> autograd computes gradients into .grad
    loss.backward()

    # 4) update parameters (no_grad: don't track this arithmetic)
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # 5) reset gradients to zero for the next epoch
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    # report progress
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.7456532081533994
Epoch: 2, Loss: 0.7386210355676442
Epoch: 3, Loss: 0.7317492176834806
Epoch: 4, Loss: 0.7250347393221332
Epoch: 5, Loss: 0.7184746457803837
Epoch: 6, Loss: 0.7120660401574253
Epoch: 7, Loss: 0.7058060807881513
Epoch: 8, Loss: 0.6996919787856546
Epoch: 9, Loss: 0.6937209956947735
Epoch: 10, Loss: 0.687890441257703
Epoch: 11, Loss: 0.6821976712920066
Epoch: 12, Loss: 0.676640085680803
Epoch: 13, Loss: 0.6712151264744007
Epoch: 14, Loss: 0.665920276102255
Epoch: 15, Loss: 0.6607530556937711
Epoch: 16, Loss: 0.6557110235061847
Epoch: 17, Loss: 0.6507917734574857
Epoch: 18, Loss: 0.6459929337621347
Epoch: 19, Loss: 0.6413121656671116
Epoch: 20, Loss: 0.636747162285667
Epoch: 21, Loss: 0.6322956475259767
Epoch: 22, Loss: 0.6279553751117528
Epoch: 23, Loss: 0.623724127691732
Epoch: 24, Loss: 0.6195997160348419
Epoch: 25, Loss: 0.6155799783077329
Epoch: 26, Loss: 0.6116627794312754
Epoch: 27, Loss: 0.6078460105125364
Epoch: 28, Loss: 0.6041275883486894
Epoch:

In [24]:
# Peek at the learned bias (it started at 0 and moved during training)
model.bias

tensor([-0.4840], dtype=torch.float64, requires_grad=True)

In [25]:
model.weights

tensor([[0.1037],
        [0.5019],
        [0.3293],
        [0.1331],
        [0.1547],
        [0.2691],
        [0.5181],
        [0.5637]], dtype=torch.float64, requires_grad=True)

## 7️⃣ Evaluation

Now we check the model on the **test set** — patients it has never seen. Two important details:

- **`torch.no_grad()`** — we're only predicting, not training, so we tell PyTorch to skip gradient tracking (faster + less memory).
- **Threshold `0.5`** — the model outputs a *probability*. We convert it to a hard `0`/`1` decision: probability `> 0.5` → predict diabetes.

> 💡 **Hint:** we reshape the target with `.view(-1, 1)` so it lines up with `y_pred`'s `(N, 1)` shape before comparing. Try a threshold of `0.3` or `0.7` and see how accuracy — and the trade-off between false positives / false negatives — shifts.

In [26]:
# model evaluation on the unseen test set
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)          # probabilities, shape (N, 1)
    y_pred = (y_pred > 0.5).float()                # 0.5 threshold -> hard 0/1 labels

    # reshape the true labels to (N, 1) so shapes match before comparing
    y_true = y_test_tensor.view(-1, 1).float()

    accuracy = (y_pred == y_true).float().mean()
    print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.6818181872367859


## 🎯 Recap & things to try

**What you built:** a complete training pipeline — load → split → scale → tensor-ize → model → train loop → evaluate — using only PyTorch tensors and autograd. No `nn.Module`, no optimizer, no `nn.BCELoss`. You did the backprop plumbing yourself!

**The mental model to keep:**
> forward → loss → `backward()` → update → **zero grad** → repeat

### 🧪 Exercises
1. **Learning-rate sweep.** Set `learning_rate` to `0.01`, `0.1`, and `1.0`. Which converges fastest? Does any diverge?
2. **More epochs.** Bump `epochs` to `200`. Does the loss keep dropping or flatten out?
3. **Threshold tuning.** In evaluation, try thresholds `0.3` and `0.7`. In a medical setting, is a *false negative* (missing a diabetic patient) worse than a *false positive*?
4. **Track accuracy over time.** Compute test accuracy every 10 epochs and print it alongside the loss.
5. **Level up (stretch).** Rewrite `MySimpleNN` using `torch.nn.Linear`, `torch.nn.BCELoss`, and `torch.optim.SGD`. Notice how much boilerplate disappears — that's the payoff for understanding the manual version first.